# Store Data - Simple Baseline

- [Forecasting with ML and Stuff](https://www.kaggle.com/code/ryanholbrook/forecasting-with-machine-learning/tutorial)
- [Kaggle Page](https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data)


In [ ]:
import torch
from torch import nn

from time_series.store_sales import HOLIDAY_FEATURE_COLS, STORE_FEATURE_COLS, StoreData

# Notes

From the information
- Wages in the public sector are paid every two weeks on the 15 th and on the last day of the month. Supermarket sales could be affected by this.
- A magnitude 7.8 earthquake struck Ecuador on April 16, 2016. People rallied in relief efforts donating water and other first need products which greatly affected supermarket sales for several weeks after the earthquake.

# Load and Explore Data

In [ ]:
store_data = StoreData(
    include_oil=True,
    include_onpromotion=True,
    store_feature_cols=list(STORE_FEATURE_COLS),
    holiday_features=list(HOLIDAY_FEATURE_COLS),
)
store_data.train

In [ ]:
store_data.oil_tensor

In [ ]:
thing = torch.stack(
    [
        store_data.sales_tensor,
        store_data.promotion_tensor,
        store_data.oil_tensor.view(-1, 1, 1).expand(*store_data.sales_tensor.shape),
    ],
    dim=-1,
).unsqueeze(0)

thing.shape

In [ ]:
flat_thing = thing.permute(0, 3, 2, 1, 4).flatten(0, 2)
flat_thing.shape

In [ ]:
flat_thing.reshape(-1, *thing.shape[1:4], 3).shape

In [ ]:
model_thing = nn.TransformerEncoder(
    nn.TransformerEncoderLayer(
        8,
        4,
        dim_feedforward=512,
        batch_first=True,
    ),
    num_layers=12,
)
model_thing(torch.rand((1, 30, 8))).shape